Train

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os

# ------------------ Hyperparameters ------------------
FS = 250
SEQ_LEN_SEC = 10
SEQ_LEN = SEQ_LEN_SEC * FS

HIDDEN_SIZE = 128
NUM_LAYERS = 2
BATCH_SIZE = 32          # Now possible because sequences are shorter
EPOCHS = 30
LR = 0.001

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------ Dataset ------------------
class ECGDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx][:, None]
        return x, x

# ------------------ LSTM Autoencoder ------------------
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS):
        super().__init__()
        self.encoder = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, input_size, num_layers, batch_first=True)

    def forward(self, x):
        enc_out, _ = self.encoder(x)
        dec_out, _ = self.decoder(enc_out)
        return dec_out

# ------------------ Load NSR Windows ------------------
X_nsr = np.load("/content/drive/MyDrive/Models/nsr_100s/X_nsr_final.npy")

dataset = ECGDataset(X_nsr)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ------------------ Training Setup ------------------
model = LSTMAutoencoder().to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# ------------------ Training Loop ------------------
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_x, batch_y in dataloader:
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)

    avg_loss = total_loss / len(dataset)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.6f}")

# ------------------ Save Model ------------------
MODEL_DIR = "/content/drive/MyDrive/Models/NSR/"
os.makedirs(MODEL_DIR, exist_ok=True)

torch.save(
    model.state_dict(),
    os.path.join(MODEL_DIR, "LSTM_NSR_autoencoder_10s.pth")
)

print("10 second NSR LSTM autoencoder trained and saved.")


Test

In [ ]:
import wfdb
import torch
import torch.nn as nn
import numpy as np
import os
from scipy.signal import detrend, savgol_filter

# ------------------ Configuration ------------------
FS = 250
SEQ_LEN_SEC = 10
SEQ_LEN = SEQ_LEN_SEC * FS
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Thresholds
ERROR_THRESHOLD = 0.30
CONSECUTIVE_SEC = 10
WARMUP_SEC = 30         # SKIP the first 30 seconds (Filter settling time)

CUDB_PATH = "/content/drive/MyDrive/ECG_Datasets/CUDB"
MODEL_PATH = "/content/drive/MyDrive/Models/NSR/LSTM_NSR_autoencoder_10s.pth"

# ------------------ LSTM Autoencoder ------------------
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=2):
        super().__init__()
        self.encoder = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, input_size, num_layers, batch_first=True)

    def forward(self, x):
        enc_out, _ = self.encoder(x)
        dec_out, _ = self.decoder(enc_out)
        return dec_out

# ------------------ Load Model ------------------
model = LSTMAutoencoder().to(DEVICE)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print(f"Model loaded from {MODEL_PATH}")
else:
    print(f"Error: Model path not found at {MODEL_PATH}")
model.eval()

# ------------------ 1. CLEANING ------------------
def clean_signal(window):
    # FIX: Replace NaNs with 0.0
    sig = np.array(window, dtype=np.float32)
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)

    # A. Remove Sweat (Detrend)
    sig = detrend(sig, type='linear')

    # B. Remove Fuzz (Smoothing)
    try:
        sig = savgol_filter(sig, window_length=11, polyorder=3)
    except:
        pass
    return sig

# ------------------ 2. GATEKEEPER ------------------
def is_mechanically_sound(window):
    if np.std(window) < 0.001: return False # Flatline
    if np.mean(np.abs(window) > 3.0) > 0.1: return False # Saturation
    return True

# ------------------ 3. RECONSTRUCTION ------------------
def get_error_and_morphology(window):
    # FIX: ONLY MEAN SUBTRACTION (No division by STD)
    # This preserves the original Millivolt amplitude your model was trained on.
    mu = np.mean(window)
    norm_window = window - mu

    with torch.no_grad():
        x = torch.tensor(norm_window, dtype=torch.float32, device=DEVICE)
        x = x.unsqueeze(0).unsqueeze(-1)
        recon = model(x)
        recon_np = recon.squeeze().cpu().numpy()
        input_np = x.squeeze().cpu().numpy()

    mse = ((recon_np - input_np) ** 2).mean()

    # --- MORPHOLOGY CHECK ---
    if mse > ERROR_THRESHOLD:
        diff = np.abs(input_np - recon_np)

        # We use a fixed voltage threshold (e.g., 0.5mV) to find the QRS
        # instead of Z-score, since we are now in raw amplitude space.
        active_mask = np.abs(input_np) > 0.4

        if np.sum(active_mask) > 0:
            error_in_qrs = np.mean(diff[active_mask])
            error_total = np.mean(diff)
            is_structural = error_in_qrs > error_total
        else:
            is_structural = False

        return mse, is_structural

    return mse, True

# ------------------ 4. PREDICT ------------------
def detect_vtac_start_clinical(ecg):
    n_samples = len(ecg)
    n_seconds = n_samples // FS

    ecg = np.nan_to_num(ecg, nan=0.0)
    error_history = []

    # FIX: START LOOP AFTER WARMUP
    start_sec = max(SEQ_LEN_SEC, WARMUP_SEC)

    # Pad history for the warmup period so indices match
    for _ in range(start_sec):
        error_history.append(0.0)

    for sec in range(start_sec, n_seconds):
        start = (sec - SEQ_LEN_SEC) * FS
        end = start + SEQ_LEN
        if end > n_samples: break

        raw_window = ecg[start:end]
        clean_window = clean_signal(raw_window)

        if not is_mechanically_sound(clean_window):
            error_history.append(0.0)
            continue

        mse, is_structural = get_error_and_morphology(clean_window)

        if mse > ERROR_THRESHOLD and not is_structural:
            mse = 0.0

        error_history.append(mse)

    error_history = np.array(error_history)

    for i in range(len(error_history) - CONSECUTIVE_SEC + 1):
        # Skip if we are still in the padding/warmup zone
        if i < WARMUP_SEC: continue

        recent_errors = error_history[i : i + CONSECUTIVE_SEC]

        if np.all(recent_errors > ERROR_THRESHOLD):
            # Check for sudden jump
            if i > 5:
                prev_baseline = np.mean(error_history[i-5 : i])
                current_level = np.mean(recent_errors)
                if (current_level - prev_baseline) > 0.5:
                    continue

            return i + SEQ_LEN_SEC

    return None

# ==================================================
#           MAIN EVALUATION LOOP
# ==================================================

print(f"{'Record':<10} | {'Annotated':<10} | {'Predicted':<10} | {'Diff (s)':<10} | {'Status'}")
print("-" * 65)

results = []

for idx in range(1, 2):
    rec_name = f"cu{idx:02d}"

    try:
        record = wfdb.rdrecord(os.path.join(CUDB_PATH, rec_name))
        ann = wfdb.rdann(os.path.join(CUDB_PATH, rec_name), 'atr')
    except Exception:
        continue

    ecg = record.p_signal[:, 0]

    vtac_annot_sec = None
    for sym, samp in zip(ann.symbol, ann.sample):
        if sym == '[':
            vtac_annot_sec = samp // FS
            break

    if vtac_annot_sec is None: continue

    vtac_pred_sec = detect_vtac_start_clinical(ecg)

    if vtac_pred_sec is None:
        print(f"{rec_name:<10} | {vtac_annot_sec:<10} | {'---':<10} | {'---':<10} | MISSED")
    else:
        diff_sec = vtac_pred_sec - vtac_annot_sec
        results.append(diff_sec)

        status = "EARLY" if diff_sec < 0 else "LATE"
        print(f"{rec_name:<10} | {vtac_annot_sec:<10} | {vtac_pred_sec:<10} | {diff_sec:<10.2f} | {status}")

# ==================================================
#               FINAL SUMMARY
# ==================================================
print("-" * 65)
if len(results) == 0:
    print("No valid detections.")
else:
    mean_diff = np.mean(results)
    std_diff = np.std(results)
    print(f"Total Detected: {len(results)}")
    print(f"Mean Prediction Time: {mean_diff:.2f} seconds ({mean_diff/60:.2f} min)")

In [ ]:
import wfdb
import torch
import torch.nn as nn
import numpy as np
import os
from scipy.signal import detrend, savgol_filter

# ------------------ Configuration ------------------
FS = 250
SEQ_LEN_SEC = 10
SEQ_LEN = SEQ_LEN_SEC * FS
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Clinical Thresholds
ERROR_THRESHOLD = 0.30
CONSECUTIVE_SEC = 10
WARMUP_SEC = 30         # Filter settling time

# Evaluation Settings
MAX_MINUTES = 30        # Analyze first 30 mins of each record
NSRDB_PATH = "/content/drive/MyDrive/nsrdb"
MODEL_PATH = "/content/drive/MyDrive/Models/NSR/LSTM_NSR_autoencoder_10s.pth"

# ------------------ LSTM Autoencoder ------------------
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=2):
        super().__init__()
        self.encoder = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, input_size, num_layers, batch_first=True)

    def forward(self, x):
        enc_out, _ = self.encoder(x)
        dec_out, _ = self.decoder(enc_out)
        return dec_out

# ------------------ Load Model ------------------
model = LSTMAutoencoder().to(DEVICE)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print(f"Model loaded from {MODEL_PATH}")
else:
    print(f"Error: Model path not found at {MODEL_PATH}")
model.eval()

# ------------------ 1. CLEANING ------------------
def clean_signal(window):
    sig = np.array(window, dtype=np.float32)
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
    sig = detrend(sig, type='linear')
    try:
        sig = savgol_filter(sig, window_length=11, polyorder=3)
    except:
        pass
    return sig

# ------------------ 2. GATEKEEPER ------------------
def is_mechanically_sound(window):
    if np.std(window) < 0.001: return False
    if np.mean(np.abs(window) > 3.0) > 0.1: return False
    return True

# ------------------ 3. RECONSTRUCTION ------------------
def get_error_and_morphology(window):
    mu = np.mean(window)
    norm_window = window - mu

    with torch.no_grad():
        x = torch.tensor(norm_window, dtype=torch.float32, device=DEVICE)
        x = x.unsqueeze(0).unsqueeze(-1)
        recon = model(x)
        recon_np = recon.squeeze().cpu().numpy()
        input_np = x.squeeze().cpu().numpy()

    mse = ((recon_np - input_np) ** 2).mean()

    if mse > ERROR_THRESHOLD:
        diff = np.abs(input_np - recon_np)
        active_mask = np.abs(input_np) > 0.4

        if np.sum(active_mask) > 0:
            error_in_qrs = np.mean(diff[active_mask])
            error_total = np.mean(diff)
            is_structural = error_in_qrs > error_total
        else:
            is_structural = False
        return mse, is_structural

    return mse, True

# ------------------ 4. PREDICT & COUNT ------------------
def count_false_positives(ecg):
    """
    Scans the entire signal and counts DISTINCT false positive events.
    """
    n_samples = len(ecg)
    n_seconds = n_samples // FS

    ecg = np.nan_to_num(ecg, nan=0.0)
    error_history = []

    # Warmup padding
    start_sec = max(SEQ_LEN_SEC, WARMUP_SEC)
    for _ in range(start_sec):
        error_history.append(0.0)

    # --- Pass 1: Calculate Errors for the whole file ---
    for sec in range(start_sec, n_seconds):
        start = (sec - SEQ_LEN_SEC) * FS
        end = start + SEQ_LEN
        if end > n_samples: break

        raw_window = ecg[start:end]
        clean_window = clean_signal(raw_window)

        if not is_mechanically_sound(clean_window):
            error_history.append(0.0)
            continue

        mse, is_structural = get_error_and_morphology(clean_window)

        if mse > ERROR_THRESHOLD and not is_structural:
            mse = 0.0

        error_history.append(mse)

    # --- Pass 2: Count Distinct Events ---
    error_history = np.array(error_history)
    fp_events = 0
    i = WARMUP_SEC

    # We use a while loop so we can skip ahead when an event is found
    while i < len(error_history) - CONSECUTIVE_SEC + 1:

        recent_errors = error_history[i : i + CONSECUTIVE_SEC]

        # Check for persistent error
        if np.all(recent_errors > ERROR_THRESHOLD):

            # Artifact Rejection (Sudden Jump check)
            is_artifact = False
            if i > 5:
                prev_baseline = np.mean(error_history[i-5 : i])
                current_level = np.mean(recent_errors)
                if (current_level - prev_baseline) > 0.5:
                    is_artifact = True

            if not is_artifact:
                # Valid FP Event Found!
                fp_events += 1

                # SKIP LOGIC: Jump forward to avoid counting this same event multiple times.
                # We skip the length of the event (CONSECUTIVE_SEC)
                i += CONSECUTIVE_SEC
                continue

        # If no event, move to next second
        i += 1

    return fp_events

# ==================================================
#           NSRDB FALSE POSITIVE TEST
# ==================================================

print(f"{'Record':<10} | {'Duration (m)':<12} | {'FP Events':<10}")
print("-" * 45)

total_records = 0
total_fp_events = 0
total_hours_analyzed = 0.0

try:
    files = sorted([f.replace('.hea', '') for f in os.listdir(NSRDB_PATH) if f.endswith('.hea')])
except FileNotFoundError:
    print(f"Error: NSRDB path not found at {NSRDB_PATH}")
    files = []

for rec_name in files:
    try:
        record = wfdb.rdrecord(os.path.join(NSRDB_PATH, rec_name), sampto=MAX_MINUTES*60*FS)
        ecg = record.p_signal[:, 0]
    except Exception:
        continue

    total_records += 1

    # Calculate actual duration in hours (some records might be shorter than MAX_MINUTES)
    duration_hours = (len(ecg) / FS) / 3600.0
    total_hours_analyzed += duration_hours

    # Count Events
    events = count_false_positives(ecg)
    total_fp_events += events

    status_str = "CLEAN" if events == 0 else str(events)
    print(f"{rec_name:<10} | {duration_hours*60:<12.1f} | {status_str:<10}")

# ==================================================
#               FINAL SUMMARY
# ==================================================
print("-" * 60)
if total_records > 0 and total_hours_analyzed > 0:
    fp_per_hour = total_fp_events / total_hours_analyzed

    print(f"Total Records:      {total_records}")
    print(f"Total Hours:        {total_hours_analyzed:.2f}")
    print(f"Total FP Events:    {total_fp_events}")
    print(f"False Pos / Hour:   {fp_per_hour:.2f}")

    print("-" * 60)
    if fp_per_hour < 1.0:
        print("[EXCELLENT] Clinical grade performance (< 1 FP/hour).")
    elif fp_per_hour < 5.0:
        print("[GOOD] Acceptable for ward monitoring, maybe noisy for ICU.")
    else:
        print("[HIGH] Too many alarms. Needs threshold tuning.")
else:
    print("No records processed.")

In [ ]:
import wfdb
import os

# Define where you want to save it
VFDB_PATH = "/content/drive/MyDrive/ECG_Datasets/VFDB"

# Create directory if it doesn't exist
if not os.path.exists(VFDB_PATH):
    os.makedirs(VFDB_PATH)

print(f"Downloading VFDB to {VFDB_PATH}...")

try:
    # 'vfdb' is the PhysioNet identifier
    wfdb.dl_database('vfdb', VFDB_PATH)
    print("Download complete.")
except Exception as e:
    print(f"Error downloading: {e}")

In [ ]:
import wfdb
import torch
import torch.nn as nn
import numpy as np
import os
from scipy.signal import detrend, savgol_filter

# ------------------ Configuration ------------------
FS = 250
SEQ_LEN_SEC = 10
SEQ_LEN = SEQ_LEN_SEC * FS
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Clinical Thresholds
ERROR_THRESHOLD = 0.30
CONSECUTIVE_SEC = 10
WARMUP_SEC = 30

# Dataset Paths
VFDB_PATH = "/content/drive/MyDrive/ECG_Datasets/VFDB"
MODEL_PATH = "/content/drive/MyDrive/Models/NSR/LSTM_NSR_autoencoder_10s.pth"

# ------------------ LSTM Autoencoder ------------------
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=2):
        super().__init__()
        self.encoder = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, input_size, num_layers, batch_first=True)

    def forward(self, x):
        enc_out, _ = self.encoder(x)
        dec_out, _ = self.decoder(enc_out)
        return dec_out

# ------------------ Load Model ------------------
model = LSTMAutoencoder().to(DEVICE)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print(f"Model loaded from {MODEL_PATH}")
else:
    print(f"Error: Model path not found at {MODEL_PATH}")
model.eval()

# ------------------ 1. CLEANING ------------------
def clean_signal(window):
    sig = np.array(window, dtype=np.float32)
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
    sig = detrend(sig, type='linear')
    try:
        sig = savgol_filter(sig, window_length=11, polyorder=3)
    except:
        pass
    return sig

# ------------------ 2. GATEKEEPER ------------------
def is_mechanically_sound(window):
    if np.std(window) < 0.001: return False
    if np.mean(np.abs(window) > 3.0) > 0.1: return False
    return True

# ------------------ 3. RECONSTRUCTION ------------------
def get_error_and_morphology(window):
    mu = np.mean(window)
    norm_window = window - mu

    with torch.no_grad():
        x = torch.tensor(norm_window, dtype=torch.float32, device=DEVICE)
        x = x.unsqueeze(0).unsqueeze(-1)
        recon = model(x)
        recon_np = recon.squeeze().cpu().numpy()
        input_np = x.squeeze().cpu().numpy()

    mse = ((recon_np - input_np) ** 2).mean()

    if mse > ERROR_THRESHOLD:
        diff = np.abs(input_np - recon_np)
        active_mask = np.abs(input_np) > 0.4

        if np.sum(active_mask) > 0:
            error_in_qrs = np.mean(diff[active_mask])
            error_total = np.mean(diff)
            is_structural = error_in_qrs > error_total
        else:
            is_structural = False

        return mse, is_structural

    return mse, True

# ------------------ 4. PREDICT ------------------
def detect_vtac_start_clinical(ecg):
    n_samples = len(ecg)
    n_seconds = n_samples // FS

    ecg = np.nan_to_num(ecg, nan=0.0)
    error_history = []

    start_sec = max(SEQ_LEN_SEC, WARMUP_SEC)
    for _ in range(start_sec):
        error_history.append(0.0)

    for sec in range(start_sec, n_seconds):
        start = (sec - SEQ_LEN_SEC) * FS
        end = start + SEQ_LEN
        if end > n_samples: break

        raw_window = ecg[start:end]
        clean_window = clean_signal(raw_window)

        if not is_mechanically_sound(clean_window):
            error_history.append(0.0)
            continue

        mse, is_structural = get_error_and_morphology(clean_window)

        if mse > ERROR_THRESHOLD and not is_structural:
            mse = 0.0

        error_history.append(mse)

    error_history = np.array(error_history)

    for i in range(len(error_history) - CONSECUTIVE_SEC + 1):
        if i < WARMUP_SEC: continue

        recent_errors = error_history[i : i + CONSECUTIVE_SEC]

        if np.all(recent_errors > ERROR_THRESHOLD):
            if i > 5:
                prev_baseline = np.mean(error_history[i-5 : i])
                current_level = np.mean(recent_errors)
                if (current_level - prev_baseline) > 0.5:
                    continue

            return i + SEQ_LEN_SEC

    return None

# ==================================================
#           MAIN EVALUATION LOOP (VFDB)
# ==================================================

print(f"{'Record':<10} | {'Annotated':<10} | {'Predicted':<10} | {'Diff (s)':<10} | {'Status'}")
print("-" * 65)

results = []

# 1. FILE CHECK
try:
    if not os.path.exists(VFDB_PATH):
        print(f"CRITICAL: VFDB Folder not found at {VFDB_PATH}")
        files = []
    else:
        files = sorted([f.replace('.dat', '') for f in os.listdir(VFDB_PATH) if f.endswith('.dat')])
        if len(files) == 0:
            print(f"CRITICAL: No .dat files found in {VFDB_PATH}. Did the download finish?")
except Exception as e:
    print(f"Error reading directory: {e}")
    files = []

for rec_name in files:
    try:
        record = wfdb.rdrecord(os.path.join(VFDB_PATH, rec_name))
        ann = wfdb.rdann(os.path.join(VFDB_PATH, rec_name), 'atr')
    except Exception:
        # print(f"Skipping {rec_name}, load failed.")
        continue

    # VFDB has 2 leads. Lead 0 is usually ECG.
    ecg = record.p_signal[:, 0]

    # Fallback: If Lead 0 is flat, try Lead 1
    if np.std(ecg) < 0.05 and record.p_signal.shape[1] > 1:
        ecg = record.p_signal[:, 1]

    # -------- UPDATED VTAC ANNOTATION LOGIC --------
    vtac_annot_sec = None

    # VFDB uses text notes like "(VT" or "(VFL" in addition to symbols
    for i in range(len(ann.sample)):
        sym = ann.symbol[i]

        # Check text note if available
        aux = ann.aux_note[i] if hasattr(ann, 'aux_note') else ""

        # Condition 1: Strict VF symbol '['
        if sym == '[':
            vtac_annot_sec = ann.sample[i] // FS
            break

        # Condition 2: Rhythm Change '+' with Text Note
        if sym == '+' and ('(VT' in aux or '(VFL' in aux):
            vtac_annot_sec = ann.sample[i] // FS
            break

    if vtac_annot_sec is None:
        # print(f"Skipping {rec_name}, no VT/VF annotation found.")
        continue

    # -------- Clinical Detection --------
    vtac_pred_sec = detect_vtac_start_clinical(ecg)

    # -------- Result Logging --------
    if vtac_pred_sec is None:
        print(f"{rec_name:<10} | {vtac_annot_sec:<10} | {'---':<10} | {'---':<10} | MISSED")
    else:
        diff_sec = vtac_pred_sec - vtac_annot_sec
        results.append(diff_sec)

        status = "EARLY" if diff_sec < 0 else "LATE"
        print(f"{rec_name:<10} | {vtac_annot_sec:<10} | {vtac_pred_sec:<10} | {diff_sec:<10.2f} | {status}")

# ==================================================
#               FINAL SUMMARY
# ==================================================
print("-" * 65)
if len(results) == 0:
    print("No valid detections.")
else:
    mean_diff = np.mean(results)
    std_diff = np.std(results)
    print(f"Total Detected: {len(results)}")
    print(f"Mean Prediction Time: {mean_diff:.2f} seconds ({mean_diff/60:.2f} min)")
    print(f"Standard Deviation: {std_diff:.2f} seconds")

In [ ]:
import wfdb
import os

FANTASIA_PATH = "/content/drive/MyDrive/ECG_Datasets/Fantasia"

if not os.path.exists(FANTASIA_PATH):
    os.makedirs(FANTASIA_PATH)

print(f"Downloading Fantasia Database to {FANTASIA_PATH}...")

try:
    # 'fantasia' is the PhysioNet identifier
    wfdb.dl_database('fantasia', FANTASIA_PATH)
    print("Download complete.")
except Exception as e:
    print(f"Error downloading: {e}")

In [ ]:
import wfdb
import torch
import torch.nn as nn
import numpy as np
import os
from scipy.signal import detrend, savgol_filter

# ------------------ Configuration ------------------
FS = 250
SEQ_LEN_SEC = 10
SEQ_LEN = SEQ_LEN_SEC * FS
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Clinical Thresholds
ERROR_THRESHOLD = 0.30
CONSECUTIVE_SEC = 10
WARMUP_SEC = 30

# Dataset Settings
MAX_MINUTES = 30        # Analyze first 30 mins
FANTASIA_PATH = "/content/drive/MyDrive/ECG_Datasets/Fantasia"
MODEL_PATH = "/content/drive/MyDrive/Models/NSR/LSTM_NSR_autoencoder_10s.pth"

# ------------------ LSTM Autoencoder ------------------
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=2):
        super().__init__()
        self.encoder = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, input_size, num_layers, batch_first=True)

    def forward(self, x):
        enc_out, _ = self.encoder(x)
        dec_out, _ = self.decoder(enc_out)
        return dec_out

# ------------------ Load Model ------------------
model = LSTMAutoencoder().to(DEVICE)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print(f"Model loaded from {MODEL_PATH}")
else:
    print(f"Error: Model path not found at {MODEL_PATH}")
model.eval()

# ------------------ 1. CLEANING ------------------
def clean_signal(window):
    sig = np.array(window, dtype=np.float32)
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
    sig = detrend(sig, type='linear')
    try:
        sig = savgol_filter(sig, window_length=11, polyorder=3)
    except:
        pass
    return sig

# ------------------ 2. GATEKEEPER ------------------
def is_mechanically_sound(window):
    if np.std(window) < 0.001: return False
    if np.mean(np.abs(window) > 3.0) > 0.1: return False
    return True

# ------------------ 3. RECONSTRUCTION ------------------
def get_error_and_morphology(window):
    mu = np.mean(window)
    norm_window = window - mu

    with torch.no_grad():
        x = torch.tensor(norm_window, dtype=torch.float32, device=DEVICE)
        x = x.unsqueeze(0).unsqueeze(-1)
        recon = model(x)
        recon_np = recon.squeeze().cpu().numpy()
        input_np = x.squeeze().cpu().numpy()

    mse = ((recon_np - input_np) ** 2).mean()

    if mse > ERROR_THRESHOLD:
        diff = np.abs(input_np - recon_np)
        active_mask = np.abs(input_np) > 0.4

        if np.sum(active_mask) > 0:
            error_in_qrs = np.mean(diff[active_mask])
            error_total = np.mean(diff)
            is_structural = error_in_qrs > error_total
        else:
            is_structural = False

        return mse, is_structural

    return mse, True

# ------------------ 4. COUNT FALSE POSITIVES ------------------
def count_false_positives(ecg):
    n_samples = len(ecg)
    n_seconds = n_samples // FS

    ecg = np.nan_to_num(ecg, nan=0.0)
    error_history = []

    start_sec = max(SEQ_LEN_SEC, WARMUP_SEC)
    for _ in range(start_sec):
        error_history.append(0.0)

    # Pass 1: Calc Errors
    for sec in range(start_sec, n_seconds):
        start = (sec - SEQ_LEN_SEC) * FS
        end = start + SEQ_LEN
        if end > n_samples: break

        raw_window = ecg[start:end]
        clean_window = clean_signal(raw_window)

        if not is_mechanically_sound(clean_window):
            error_history.append(0.0)
            continue

        mse, is_structural = get_error_and_morphology(clean_window)

        if mse > ERROR_THRESHOLD and not is_structural:
            mse = 0.0

        error_history.append(mse)

    # Pass 2: Count Events
    error_history = np.array(error_history)
    fp_events = 0
    i = WARMUP_SEC

    while i < len(error_history) - CONSECUTIVE_SEC + 1:
        recent_errors = error_history[i : i + CONSECUTIVE_SEC]

        if np.all(recent_errors > ERROR_THRESHOLD):
            is_artifact = False
            if i > 5:
                prev_baseline = np.mean(error_history[i-5 : i])
                current_level = np.mean(recent_errors)
                if (current_level - prev_baseline) > 0.5:
                    is_artifact = True

            if not is_artifact:
                fp_events += 1
                i += CONSECUTIVE_SEC
                continue

        i += 1

    return fp_events

# ==================================================
#           FANTASIA EVALUATION LOOP
# ==================================================

print(f"{'Record':<10} | {'Group':<10} | {'FP Events':<10}")
print("-" * 45)

total_records = 0
total_fp_events = 0
total_hours_analyzed = 0.0

try:
    if not os.path.exists(FANTASIA_PATH):
        print("Error: Fantasia folder not found.")
        files = []
    else:
        files = sorted([f.replace('.dat', '') for f in os.listdir(FANTASIA_PATH) if f.endswith('.dat')])
except Exception as e:
    print(f"Error: {e}")
    files = []

for rec_name in files:
    try:
        # Load record
        record = wfdb.rdrecord(os.path.join(FANTASIA_PATH, rec_name), sampto=MAX_MINUTES*60*FS)

        # Fantasia has 'ECG' and 'RESP'. We need to find the ECG channel.
        ecg_idx = 0
        if 'ECG' in record.sig_name:
            ecg_idx = record.sig_name.index('ECG')
        else:
            # Heuristic: ECG usually has higher variance than Respiration
            std0 = np.std(record.p_signal[:, 0])
            std1 = np.std(record.p_signal[:, 1])
            ecg_idx = 0 if std0 > std1 else 1

        ecg = record.p_signal[:, ecg_idx]

    except Exception:
        continue

    total_records += 1

    duration_hours = (len(ecg) / FS) / 3600.0
    total_hours_analyzed += duration_hours

    # Identify Group based on filename (f1y = young, f1o = old)
    group = "Young" if 'y' in rec_name else "Elderly"

    # Count Events
    events = count_false_positives(ecg)
    total_fp_events += events

    status_str = "CLEAN" if events == 0 else str(events)
    print(f"{rec_name:<10} | {group:<10} | {status_str:<10}")

# ==================================================
#               FINAL SUMMARY
# ==================================================


print("-" * 60)
if total_records > 0 and total_hours_analyzed > 0:
    fp_per_hour = total_fp_events / total_hours_analyzed

    print(f"Total Records:      {total_records}")
    print(f"Total Hours:        {total_hours_analyzed:.2f}")
    print(f"Total FP Events:    {total_fp_events}")
    print(f"False Pos / Hour:   {fp_per_hour:.2f}")

    print("-" * 60)
    print("Interpretation:")
    print("If 'Elderly' records have significantly more FPs than 'Young',")
    print("it means the model is sensitive to age-related benign changes.")
else:
    print("No records processed.")

In [ ]:
!pip install onnxscript
import torch
import torch.nn as nn
import os

# ------------------ Configuration ------------------
FS = 250
SEQ_LEN_SEC = 10
SEQ_LEN = SEQ_LEN_SEC * FS  # 2500
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH_PTH = "/content/drive/MyDrive/Models/NSR/LSTM_NSR_autoencoder_10s.pth"
MODEL_PATH_ONNX = "/content/drive/MyDrive/Models/NSR/LSTM_NSR_autoencoder_10s.onnx"

# ------------------ LSTM Autoencoder ------------------
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=2):
        super().__init__()
        self.encoder = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, input_size, num_layers, batch_first=True)

    def forward(self, x):
        enc_out, _ = self.encoder(x)
        dec_out, _ = self.decoder(enc_out)
        return dec_out

# ------------------ Conversion Logic ------------------
# ------------------ Conversion Logic ------------------
def convert_to_onnx():
    print("Loading PyTorch model...")

    # FIX 1: Force the export process to happen on the CPU
    # This completely bypasses the GPU/CuDNN FakeTensor memory crash
    export_device = torch.device("cpu")
    model = LSTMAutoencoder().to(export_device)

    if not os.path.exists(MODEL_PATH_PTH):
        print(f"Error: PyTorch model not found at {MODEL_PATH_PTH}")
        return

    # Load the weights and map them to the CPU
    model.load_state_dict(torch.load(MODEL_PATH_PTH, map_location=export_device))
    model.eval() # MUST be in eval mode for export

    # Create the dummy input tensor on the CPU
    print("Generating dummy input tensor (1, 2500, 1)...")
    dummy_input = torch.randn(1, SEQ_LEN, 1, device=export_device)

    print(f"Exporting model to ONNX format...")
    torch.onnx.export(
        model,
        dummy_input,
        MODEL_PATH_ONNX,
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={
            'input': {0: 'batch_size'},
            'output': {0: 'batch_size'}
        },
        # FIX 2: Explicitly disable the experimental Dynamo exporter
        # to use the stable TorchScript tracer for LSTMs.
        # (Note: If this throws an "unexpected keyword" error in your specific
        # PyTorch version, just delete this line, the CPU fix above is usually enough).
        dynamo=False
    )
    print(f"Success! ONNX model saved successfully to:\n{MODEL_PATH_ONNX}")

# Run the conversion
convert_to_onnx()

In [ ]:
!pip install wfdb onnxruntime
import wfdb
import torch
import torch.nn as nn
import numpy as np
import os
import onnxruntime as ort
from scipy.signal import detrend, savgol_filter

# ------------------ Configuration ------------------
FS = 250
SEQ_LEN_SEC = 10
SEQ_LEN = SEQ_LEN_SEC * FS
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Clinical Thresholds
ERROR_THRESHOLD = 0.30
CONSECUTIVE_SEC = 10
WARMUP_SEC = 30

# Dataset Settings
MAX_MINUTES = 30        # Analyze first 30 mins
FANTASIA_PATH = "/content/drive/MyDrive/ECG_Datasets/Fantasia"

# Model Paths
MODEL_PATH_PTH = "/content/drive/MyDrive/Models/NSR/LSTM_NSR_autoencoder_10s.pth"
MODEL_PATH_ONNX = "/content/drive/MyDrive/Models/NSR/LSTM_NSR_autoencoder_10s.onnx"

# ------------------ LSTM Autoencoder ------------------
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=2):
        super().__init__()
        self.encoder = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, input_size, num_layers, batch_first=True)

    def forward(self, x):
        enc_out, _ = self.encoder(x)
        dec_out, _ = self.decoder(enc_out)
        return dec_out

# ------------------ Load Models (PTH & ONNX) ------------------
# 1. Load PyTorch Model
model_pth = LSTMAutoencoder().to(DEVICE)
if os.path.exists(MODEL_PATH_PTH):
    model_pth.load_state_dict(torch.load(MODEL_PATH_PTH, map_location=DEVICE))
    print(f"PyTorch Model loaded from {MODEL_PATH_PTH}")
else:
    print(f"Error: PyTorch Model path not found at {MODEL_PATH_PTH}")
model_pth.eval()

# 2. Load ONNX Model
if os.path.exists(MODEL_PATH_ONNX):
    ort_session = ort.InferenceSession(MODEL_PATH_ONNX)
    input_name = ort_session.get_inputs()[0].name
    print(f"ONNX Model loaded from {MODEL_PATH_ONNX}")
else:
    print(f"Error: ONNX Model path not found at {MODEL_PATH_ONNX}")

# ------------------ 1. CLEANING ------------------
def clean_signal(window):
    sig = np.array(window, dtype=np.float32)
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
    sig = detrend(sig, type='linear')
    try:
        sig = savgol_filter(sig, window_length=11, polyorder=3)
    except:
        pass
    return sig

# ------------------ 2. GATEKEEPER ------------------
def is_mechanically_sound(window):
    if np.std(window) < 0.001: return False
    if np.mean(np.abs(window) > 3.0) > 0.1: return False
    return True

# ------------------ 3. RECONSTRUCTION (Dual Mode) ------------------
def get_error_and_morphology(window, model_type="pth"):
    mu = np.mean(window)
    norm_window = window - mu

    # Prepare NumPy array of shape (1, 2500, 1)
    x_np = np.expand_dims(np.expand_dims(norm_window, axis=0), axis=-1).astype(np.float32)

    if model_type == "pth":
        with torch.no_grad():
            x_tensor = torch.tensor(x_np, device=DEVICE)
            recon = model_pth(x_tensor)
            recon_np = recon.squeeze().cpu().numpy()
    elif model_type == "onnx":
        ort_inputs = {input_name: x_np}
        recon = ort_session.run(None, ort_inputs)[0]
        recon_np = np.squeeze(recon)

    input_np = np.squeeze(x_np)
    mse = ((recon_np - input_np) ** 2).mean()

    if mse > ERROR_THRESHOLD:
        diff = np.abs(input_np - recon_np)
        active_mask = np.abs(input_np) > 0.4

        if np.sum(active_mask) > 0:
            error_in_qrs = np.mean(diff[active_mask])
            error_total = np.mean(diff)
            is_structural = error_in_qrs > error_total
        else:
            is_structural = False

        return mse, is_structural

    return mse, True

# ------------------ 4. COUNT FALSE POSITIVES ------------------
def count_false_positives(ecg, model_type="pth"):
    n_samples = len(ecg)
    n_seconds = n_samples // FS

    ecg = np.nan_to_num(ecg, nan=0.0)
    error_history = []

    start_sec = max(SEQ_LEN_SEC, WARMUP_SEC)
    for _ in range(start_sec):
        error_history.append(0.0)

    # Pass 1: Calc Errors
    for sec in range(start_sec, n_seconds):
        start = (sec - SEQ_LEN_SEC) * FS
        end = start + SEQ_LEN
        if end > n_samples: break

        raw_window = ecg[start:end]
        clean_window = clean_signal(raw_window)

        if not is_mechanically_sound(clean_window):
            error_history.append(0.0)
            continue

        mse, is_structural = get_error_and_morphology(clean_window, model_type)

        if mse > ERROR_THRESHOLD and not is_structural:
            mse = 0.0

        error_history.append(mse)

    # Pass 2: Count Events
    error_history = np.array(error_history)
    fp_events = 0
    i = WARMUP_SEC

    while i < len(error_history) - CONSECUTIVE_SEC + 1:
        recent_errors = error_history[i : i + CONSECUTIVE_SEC]

        if np.all(recent_errors > ERROR_THRESHOLD):
            is_artifact = False
            if i > 5:
                prev_baseline = np.mean(error_history[i-5 : i])
                current_level = np.mean(recent_errors)
                if (current_level - prev_baseline) > 0.5:
                    is_artifact = True

            if not is_artifact:
                fp_events += 1
                i += CONSECUTIVE_SEC
                continue

        i += 1

    return fp_events

# ==================================================
#           FANTASIA EVALUATION LOOP (COMPARISON)
# ==================================================

print(f"{'Record':<10} | {'Group':<10} | {'FP (PTH)':<10} | {'FP (ONNX)':<10} | {'Match Status'}")
print("-" * 65)

total_records = 0
total_hours_analyzed = 0.0
total_fp_pth = 0
total_fp_onnx = 0

try:
    if not os.path.exists(FANTASIA_PATH):
        print("Error: Fantasia folder not found.")
        files = []
    else:
        files = sorted([f.replace('.dat', '') for f in os.listdir(FANTASIA_PATH) if f.endswith('.dat')])
except Exception as e:
    print(f"Error: {e}")
    files = []

for rec_name in files:
    try:
        # Load record
        record = wfdb.rdrecord(os.path.join(FANTASIA_PATH, rec_name), sampto=MAX_MINUTES*60*FS)

        # Fantasia has 'ECG' and 'RESP'. Find the ECG channel.
        ecg_idx = 0
        if 'ECG' in record.sig_name:
            ecg_idx = record.sig_name.index('ECG')
        else:
            std0 = np.std(record.p_signal[:, 0])
            std1 = np.std(record.p_signal[:, 1])
            ecg_idx = 0 if std0 > std1 else 1

        ecg = record.p_signal[:, ecg_idx]

    except Exception:
        continue

    total_records += 1
    duration_hours = (len(ecg) / FS) / 3600.0
    total_hours_analyzed += duration_hours

    # Identify Group
    group = "Young" if 'y' in rec_name else "Elderly"

    # Count Events for BOTH models
    events_pth = count_false_positives(ecg, "pth")
    events_onnx = count_false_positives(ecg, "onnx")

    total_fp_pth += events_pth
    total_fp_onnx += events_onnx

    status_str_pth = "CLEAN" if events_pth == 0 else str(events_pth)
    status_str_onnx = "CLEAN" if events_onnx == 0 else str(events_onnx)
    match_status = "✅ MATCH" if events_pth == events_onnx else "❌ MISMATCH"

    print(f"{rec_name:<10} | {group:<10} | {status_str_pth:<10} | {status_str_onnx:<10} | {match_status}")

# ==================================================
#               FINAL SUMMARY
# ==================================================

print("-" * 65)
if total_records > 0 and total_hours_analyzed > 0:
    print(f"Total Records:      {total_records}")
    print(f"Total Hours:        {total_hours_analyzed:.2f}")
    print(f"Total FP (PTH):     {total_fp_pth}")
    print(f"Total FP (ONNX):    {total_fp_onnx}")
    print("-" * 65)

    if total_fp_pth == total_fp_onnx:
        print("SUCCESS: ONNX model behavior perfectly matches PyTorch model.")
    else:
        print("WARNING: Divergence detected between ONNX and PyTorch outputs.")
else:
    print("No records processed.")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os
import time
import math
import onnxruntime as ort

# ------------------ Configuration ------------------
FS = 250
SEQ_LEN_SEC = 10
SEQ_LEN = SEQ_LEN_SEC * FS
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model Paths
MODEL_PATH_PTH = "/content/drive/MyDrive/Models/NSR/LSTM_NSR_autoencoder_10s.pth"
MODEL_PATH_ONNX = "/content/drive/MyDrive/Models/NSR/LSTM_NSR_autoencoder_10s.onnx"

# ------------------ LSTM Autoencoder ------------------
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=2):
        super().__init__()
        self.encoder = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, input_size, num_layers, batch_first=True)

    def forward(self, x):
        enc_out, _ = self.encoder(x)
        dec_out, _ = self.decoder(enc_out)
        return dec_out

# ------------------ Load Models ------------------
# Load PyTorch Model
model_pth = LSTMAutoencoder().to(DEVICE)
if os.path.exists(MODEL_PATH_PTH):
    model_pth.load_state_dict(torch.load(MODEL_PATH_PTH, map_location=DEVICE))
model_pth.eval()

# Load ONNX Model
if os.path.exists(MODEL_PATH_ONNX):
    ort_session = ort.InferenceSession(MODEL_PATH_ONNX)
    input_name = ort_session.get_inputs()[0].name

# ------------------ Real-Time Sine Wave Simulator ------------------
def run_realtime_simulation():
    print("Initializing Real-Time Sine Wave Simulator...")
    print(f"Sampling Rate: {FS} Hz")
    print(f"Window Size: {SEQ_LEN_SEC} seconds\n")

    buffer = []
    t = 0.0          # Time tracker in seconds
    freq = 1.0       # 1 Hz Sine Wave (simulating 60 BPM)

    print("Buffering the first 10 seconds (filling the window)...")

    try:
        while True:
            # 1. Generate 1 second of new data (250 samples)
            chunk = []
            for _ in range(FS):
                val = math.sin(2 * math.pi * freq * t)
                chunk.append(val)
                t += 1.0 / FS

            buffer.extend(chunk)

            # 2. Check if we have a full 10-second window
            if len(buffer) >= SEQ_LEN:
                # Extract the exact 10-second window
                window = np.array(buffer[-SEQ_LEN:], dtype=np.float32)

                # Normalize (Mean centering)
                mu = np.mean(window)
                norm_window = window - mu

                # Prepare Numpy array for ONNX and PyTorch (1, 2500, 1)
                x_np = np.expand_dims(np.expand_dims(norm_window, axis=0), axis=-1).astype(np.float32)
                input_sq = np.squeeze(x_np)

                # --- PyTorch Inference ---
                with torch.no_grad():
                    x_tensor = torch.tensor(x_np, device=DEVICE)
                    recon_pth = model_pth(x_tensor)
                    recon_pth_np = recon_pth.squeeze().cpu().numpy()
                mse_pth = ((recon_pth_np - input_sq) ** 2).mean()

                # --- ONNX Inference ---
                ort_inputs = {input_name: x_np}
                recon_onnx = ort_session.run(None, ort_inputs)[0]
                recon_onnx_np = np.squeeze(recon_onnx)
                mse_onnx = ((recon_onnx_np - input_sq) ** 2).mean()

                # Calculate absolute difference to ensure they match
                diff = abs(mse_pth - mse_onnx)
                match_status = "✅ MATCH" if diff < 1e-5 else "❌ MISMATCH"

                # 3. Print the real-time output
                print(f"Elapsed Time: {t:05.1f}s | MSE (PTH): {mse_pth:.6f} | MSE (ONNX): {mse_onnx:.6f} | Diff: {diff:.2e} | {match_status}")

                # 4. Slide the window forward by dropping the oldest 1 second (250 samples)
                buffer = buffer[FS:]

                # 5. Wait for exactly 1 second to simulate real-time ECG streaming
                time.sleep(1)

    except KeyboardInterrupt:
        print("\nSimulation stopped by user.")

# Run the simulation
run_realtime_simulation()